In [1]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP, INSTALLS, IMPORTS, DATASET LOADING
# ============================================================

# Install required packages
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub', 'timm', 'captum', 'torchmetrics', 'grad-cam',
    'tqdm', 'matplotlib', 'seaborn', 'scikit-learn', 'Pillow'], check=False)

import os, random, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
import torchmetrics

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── Constants ────────────────────────────────────────────────
LABELS = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
NUM_CLASSES = 8
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR = 1e-4
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Download Dataset ─────────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download('andrewmvd/ocular-disease-recognition-odir5k')
print(f'Dataset path: {path}')

# ── Locate CSV and Images ────────────────────────────────────
dataset_root = Path(path)
# Find annotation CSV
csv_files = list(dataset_root.rglob('*.xlsx')) + list(dataset_root.rglob('*.csv'))
print('Files found:', csv_files[:5])

# Try to read the main annotation file
anno_file = None
for f in csv_files:
    if 'train' in str(f).lower() or 'data' in str(f).lower() or 'label' in str(f).lower():
        anno_file = f
        break
if anno_file is None and csv_files:
    anno_file = csv_files[0]

print(f'Using annotation file: {anno_file}')

if str(anno_file).endswith('.xlsx'):
    df = pd.read_excel(anno_file)
else:
    df = pd.read_csv(anno_file)

print(df.head())
print(df.columns.tolist())
print(f'Shape: {df.shape}')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ============================================================
# CELL 2: DATA PREPROCESSING & DATASET CLASS
# ============================================================

# ── Parse labels from ODIR-5K format ─────────────────────────
# ODIR-5K has columns: ID, Age, Sex, Left-Fundus, Right-Fundus, Left-Diagnostic Keywords, ...
# and binary label columns N, D, G, C, A, H, M, O

# Detect label columns
label_cols = [c for c in df.columns if c in LABELS]
print(f'Label columns found: {label_cols}')

# Detect image columns
img_cols = [c for c in df.columns if 'fundus' in c.lower() or 'image' in c.lower() or 'file' in c.lower()]
print(f'Image columns found: {img_cols}')

# Find image directory
img_dirs = [d for d in dataset_root.rglob('*') if d.is_dir()]
print('Directories:', [str(d) for d in img_dirs[:10]])

# Build a unified dataframe with image paths and labels
records = []
for _, row in df.iterrows():
    for eye in ['Left-Fundus', 'Right-Fundus']:
        if eye not in df.columns:
            continue
        fname = str(row[eye]).strip()
        # Search for file
        found = list(dataset_root.rglob(fname))
        if not found:
            continue
        img_path = found[0]
        lbl = [int(row[l]) if l in row and not pd.isna(row[l]) else 0 for l in LABELS]
        records.append({'image_path': str(img_path), 'labels': lbl})

print(f'Total records with images: {len(records)}')

if len(records) == 0:
    # Fallback: scan all images and assign from dataframe differently
    all_imgs = list(dataset_root.rglob('*.jpg')) + list(dataset_root.rglob('*.png'))
    print(f'Found {len(all_imgs)} images by scan')
    # Use first N matching
    for img_path in all_imgs[:len(df)*2]:
        records.append({'image_path': str(img_path), 'labels': [0]*8})
    print('Warning: labels may not be assigned correctly; check column names.')

all_df = pd.DataFrame(records)
train_df, val_df = train_test_split(all_df, test_size=0.2, random_state=SEED)
train_df, test_df = train_test_split(train_df, test_size=0.1, random_state=SEED)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

# ── Transforms ───────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# ── Dataset ──────────────────────────────────────────────────
class ODIRDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor(row['labels'], dtype=torch.float32)
        return img, labels

train_dataset = ODIRDataset(train_df, train_transform)
val_dataset   = ODIRDataset(val_df,   val_transform)
test_dataset  = ODIRDataset(test_df,  val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Datasets and DataLoaders ready.')

# ── Shared evaluation function ────────────────────────────────
def evaluate_model(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            out = model(imgs).cpu()
            all_logits.append(out)
            all_targets.append(labels)
    logits  = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs   = 1 / (1 + np.exp(-logits))  # sigmoid
    preds   = (probs > 0.5).astype(int)

    auc_scores, f1_scores, ap_scores = [], [], []
    for i in range(NUM_CLASSES):
        if targets[:, i].sum() > 0:
            auc_scores.append(roc_auc_score(targets[:, i], probs[:, i]))
            ap_scores.append(average_precision_score(targets[:, i], probs[:, i]))
        else:
            auc_scores.append(0.0)
            ap_scores.append(0.0)
        f1_scores.append(f1_score(targets[:, i], preds[:, i], zero_division=0))

    return {
        'macro_auc': np.mean(auc_scores),
        'macro_map': np.mean(ap_scores),
        'macro_f1':  np.mean(f1_scores),
        'per_class_auc': auc_scores,
        'per_class_f1':  f1_scores,
        'per_class_map': ap_scores,
        'probs': probs,
        'targets': targets,
        'preds': preds,
    }

# ── Shared plotting function ──────────────────────────────────
def plot_results(metrics, history, model_name):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{model_name} Results', fontsize=16)

    # 1. Training curves
    ax = axes[0, 0]
    ax.plot(history['train_loss'], label='Train Loss')
    ax.plot(history['val_loss'],   label='Val Loss')
    ax.set_title('Loss Curves'); ax.set_xlabel('Epoch'); ax.legend()

    ax = axes[0, 1]
    ax.plot(history['val_auc'], label='Val AUC')
    ax.plot(history['val_f1'],  label='Val F1')
    ax.set_title('Validation Metrics'); ax.set_xlabel('Epoch'); ax.legend()

    # 2. ROC curves per class
    from sklearn.metrics import roc_curve
    ax = axes[0, 2]
    for i, lbl in enumerate(LABELS):
        if metrics['targets'][:, i].sum() > 0:
            fpr, tpr, _ = roc_curve(metrics['targets'][:, i], metrics['probs'][:, i])
            ax.plot(fpr, tpr, label=f'{lbl} ({metrics["per_class_auc"][i]:.2f})')
    ax.plot([0,1],[0,1],'k--'); ax.set_title('ROC Curves')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=7)

    # 3. Bar charts
    x = np.arange(NUM_CLASSES)
    ax = axes[1, 0]
    ax.bar(x, metrics['per_class_auc']); ax.set_xticks(x); ax.set_xticklabels(LABELS)
    ax.set_title('AUC per Class'); ax.set_ylim(0, 1)

    ax = axes[1, 1]
    ax.bar(x, metrics['per_class_f1']); ax.set_xticks(x); ax.set_xticklabels(LABELS)
    ax.set_title('F1 per Class'); ax.set_ylim(0, 1)

    ax = axes[1, 2]
    ax.bar(x, metrics['per_class_map']); ax.set_xticks(x); ax.set_xticklabels(LABELS)
    ax.set_title('mAP per Class'); ax.set_ylim(0, 1)

    plt.tight_layout()
    save_path = OUTPUT_DIR / f'{model_name}_results.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

print('Helper functions defined.')

In [ ]:
# ============================================================
# CELL 3: RESNET50 — Full Training, Validation, Plotting, Saving
# ============================================================

import torch, torchvision.models as models, torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from pathlib import Path

print(f'Device: {DEVICE}')
MODEL_NAME = 'resnet50'

# ── Build model ──────────────────────────────────────────────
model_rn50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50 = model_rn50.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_rn50.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc = 0.0
best_path = OUTPUT_DIR / 'best_resnet50.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model_rn50.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'[RN50] Epoch {epoch}/{NUM_EPOCHS} Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model_rn50(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validate
    model_rn50.eval()
    val_loss = 0.0
    all_logits, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'[RN50] Epoch {epoch}/{NUM_EPOCHS} Val', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model_rn50(imgs)
            loss = criterion(out, labels)
            val_loss += loss.item() * imgs.size(0)
            all_logits.append(out.cpu())
            all_targets.append(labels.cpu())
    val_loss /= len(val_loader.dataset)

    logits  = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs   = 1 / (1 + np.exp(-logits))
    preds   = (probs > 0.5).astype(int)

    from sklearn.metrics import roc_auc_score, f1_score
    auc_list = []
    for i in range(NUM_CLASSES):
        if targets[:, i].sum() > 0:
            auc_list.append(roc_auc_score(targets[:, i], probs[:, i]))
        else:
            auc_list.append(0.0)
    val_auc = np.mean(auc_list)
    val_f1  = np.mean([f1_score(targets[:, i], preds[:, i], zero_division=0) for i in range(NUM_CLASSES)])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)

    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model_rn50.state_dict(), best_path)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}')

# Load best & evaluate on test
model_rn50.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_metrics_rn50 = evaluate_model(model_rn50, test_loader, DEVICE)
print(f'\nResNet50 Test — AUC: {test_metrics_rn50["macro_auc"]:.4f} | mAP: {test_metrics_rn50["macro_map"]:.4f} | F1: {test_metrics_rn50["macro_f1"]:.4f}')

plot_results(test_metrics_rn50, history, 'ResNet50')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('ResNet50 Done.')

In [ ]:
# ============================================================
# CELL 4: DENSENET121 — Full Training, Validation, Plotting, Saving
# ============================================================

import torch, torchvision.models as models, torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

print(f'Device: {DEVICE}')
MODEL_NAME = 'densenet121'

model_dn121 = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121 = model_dn121.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_dn121.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc = 0.0
best_path = OUTPUT_DIR / 'best_densenet121.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    model_dn121.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'[DN121] Epoch {epoch}/{NUM_EPOCHS} Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model_dn121(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)

    model_dn121.eval()
    val_loss = 0.0
    all_logits, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'[DN121] Epoch {epoch}/{NUM_EPOCHS} Val', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model_dn121(imgs)
            loss = criterion(out, labels)
            val_loss += loss.item() * imgs.size(0)
            all_logits.append(out.cpu())
            all_targets.append(labels.cpu())
    val_loss /= len(val_loader.dataset)

    logits  = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs   = 1 / (1 + np.exp(-logits))
    preds   = (probs > 0.5).astype(int)

    auc_list = [roc_auc_score(targets[:, i], probs[:, i]) if targets[:, i].sum() > 0 else 0.0 for i in range(NUM_CLASSES)]
    val_auc = np.mean(auc_list)
    val_f1  = np.mean([f1_score(targets[:, i], preds[:, i], zero_division=0) for i in range(NUM_CLASSES)])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model_dn121.state_dict(), best_path)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}')

model_dn121.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_metrics_dn121 = evaluate_model(model_dn121, test_loader, DEVICE)
print(f'\nDenseNet121 Test — AUC: {test_metrics_dn121["macro_auc"]:.4f} | mAP: {test_metrics_dn121["macro_map"]:.4f} | F1: {test_metrics_dn121["macro_f1"]:.4f}')

plot_results(test_metrics_dn121, history, 'DenseNet121')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('DenseNet121 Done.')

In [ ]:
# ============================================================
# CELL 5: EFFICIENTNET-B0 — Full Training, Validation, Plotting, Saving
# ============================================================

import torch, torch.nn as nn, torch.optim as optim
import timm
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

print(f'Device: {DEVICE}')
MODEL_NAME = 'efficientnet_b0'

model_efn = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model_efn = model_efn.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_efn.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc = 0.0
best_path = OUTPUT_DIR / 'best_efficientnet_b0.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    model_efn.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'[EFN-B0] Epoch {epoch}/{NUM_EPOCHS} Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model_efn(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)

    model_efn.eval()
    val_loss = 0.0
    all_logits, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'[EFN-B0] Epoch {epoch}/{NUM_EPOCHS} Val', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model_efn(imgs)
            loss = criterion(out, labels)
            val_loss += loss.item() * imgs.size(0)
            all_logits.append(out.cpu())
            all_targets.append(labels.cpu())
    val_loss /= len(val_loader.dataset)

    logits  = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs   = 1 / (1 + np.exp(-logits))
    preds   = (probs > 0.5).astype(int)

    auc_list = [roc_auc_score(targets[:, i], probs[:, i]) if targets[:, i].sum() > 0 else 0.0 for i in range(NUM_CLASSES)]
    val_auc = np.mean(auc_list)
    val_f1  = np.mean([f1_score(targets[:, i], preds[:, i], zero_division=0) for i in range(NUM_CLASSES)])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model_efn.state_dict(), best_path)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}')

model_efn.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_metrics_efn = evaluate_model(model_efn, test_loader, DEVICE)
print(f'\nEfficientNet-B0 Test — AUC: {test_metrics_efn["macro_auc"]:.4f} | mAP: {test_metrics_efn["macro_map"]:.4f} | F1: {test_metrics_efn["macro_f1"]:.4f}')

plot_results(test_metrics_efn, history, 'EfficientNet_B0')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('EfficientNet-B0 Done.')

In [ ]:
# ============================================================
# CELL 6: VIT-BASE/16 — Full Training, Validation, Plotting, Saving
# ============================================================

import torch, torch.nn as nn, torch.optim as optim
import timm
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score

print(f'Device: {DEVICE}')
MODEL_NAME = 'vit_base_patch16_224'

model_vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
model_vit = model_vit.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model_vit.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc = 0.0
best_path = OUTPUT_DIR / 'best_vit_base_patch16_224.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    model_vit.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f'[ViT] Epoch {epoch}/{NUM_EPOCHS} Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model_vit(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)

    model_vit.eval()
    val_loss = 0.0
    all_logits, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'[ViT] Epoch {epoch}/{NUM_EPOCHS} Val', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model_vit(imgs)
            loss = criterion(out, labels)
            val_loss += loss.item() * imgs.size(0)
            all_logits.append(out.cpu())
            all_targets.append(labels.cpu())
    val_loss /= len(val_loader.dataset)

    logits  = torch.cat(all_logits).numpy()
    targets = torch.cat(all_targets).numpy()
    probs   = 1 / (1 + np.exp(-logits))
    preds   = (probs > 0.5).astype(int)

    auc_list = [roc_auc_score(targets[:, i], probs[:, i]) if targets[:, i].sum() > 0 else 0.0 for i in range(NUM_CLASSES)]
    val_auc = np.mean(auc_list)
    val_f1  = np.mean([f1_score(targets[:, i], preds[:, i], zero_division=0) for i in range(NUM_CLASSES)])

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model_vit.state_dict(), best_path)

    print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}')

model_vit.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_metrics_vit = evaluate_model(model_vit, test_loader, DEVICE)
print(f'\nViT-Base Test — AUC: {test_metrics_vit["macro_auc"]:.4f} | mAP: {test_metrics_vit["macro_map"]:.4f} | F1: {test_metrics_vit["macro_f1"]:.4f}')

plot_results(test_metrics_vit, history, 'ViT_Base_Patch16_224')

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('ViT-Base Done.')

In [ ]:
# ============================================================
# CELL 7: XAI HELPERS — Faithfulness Metrics (Deletion & Insertion)
# Used by ALL subsequent XAI cells
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch.nn.functional as F

def compute_faithfulness(model, img_tensor, saliency_map, class_idx, device,
                          steps=10, mode='deletion'):
    """
    Compute deletion or insertion faithfulness curve.
    img_tensor: (1, 3, H, W) normalized tensor
    saliency_map: (H, W) numpy array, higher = more important
    Returns: (step_fractions, scores)
    """
    model.eval()
    H, W = img_tensor.shape[-2], img_tensor.shape[-1]
    N = H * W

    # Flatten and sort by importance (descending)
    flat_sal = saliency_map.flatten()
    sorted_idx = np.argsort(flat_sal)[::-1]  # most important first

    # Blurred baseline for insertion
    blurred = F.avg_pool2d(img_tensor, kernel_size=11, stride=1, padding=5)

    fractions = np.linspace(0, 1, steps + 1)
    scores = []

    with torch.no_grad():
        for frac in fractions:
            k = int(frac * N)
            mask = np.zeros(N, dtype=np.float32)
            if k > 0:
                mask[sorted_idx[:k]] = 1.0
            mask_2d = torch.tensor(mask.reshape(1, 1, H, W)).to(device)

            if mode == 'deletion':
                perturbed = img_tensor.to(device) * (1 - mask_2d)
            else:  # insertion
                perturbed = blurred.to(device) * (1 - mask_2d) + img_tensor.to(device) * mask_2d

            out = model(perturbed)
            prob = torch.sigmoid(out[0, class_idx]).item()
            scores.append(prob)

    return fractions.tolist(), scores


def plot_faithfulness(del_fracs, del_scores, ins_fracs, ins_scores,
                       model_name, xai_method, class_name, save_path):
    del_auc = np.trapz(del_scores, del_fracs)
    ins_auc = np.trapz(ins_scores, ins_fracs)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{model_name} | {xai_method} | Class: {class_name}', fontsize=13)

    axes[0].plot(del_fracs, del_scores, 'r-o', markersize=4)
    axes[0].set_title(f'Deletion (AUC={del_auc:.3f})')
    axes[0].set_xlabel('Fraction of pixels removed')
    axes[0].set_ylabel('Probability')

    axes[1].plot(ins_fracs, ins_scores, 'g-o', markersize=4)
    axes[1].set_title(f'Insertion (AUC={ins_auc:.3f})')
    axes[1].set_xlabel('Fraction of pixels revealed')
    axes[1].set_ylabel('Probability')

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved faithfulness: {save_path}')
    return del_auc, ins_auc


def get_sample_image(dataset, idx=0):
    """Return (img_tensor (1,3,H,W), orig_pil, label_vec)"""
    img_tensor, labels = dataset[idx]
    img_tensor = img_tensor.unsqueeze(0)  # (1,3,H,W)
    # Denormalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    orig = img_tensor.squeeze(0) * std + mean
    orig = orig.permute(1, 2, 0).numpy().clip(0, 1)
    return img_tensor, orig, labels


def overlay_heatmap(orig_img, heatmap, alpha=0.5):
    """Overlay a (H,W) heatmap onto orig_img (H,W,3) numpy [0,1]."""
    import matplotlib.cm as cm
    heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap_color = cm.jet(heatmap_norm)[:, :, :3]  # (H,W,3)
    overlay = alpha * heatmap_color + (1 - alpha) * orig_img
    return overlay.clip(0, 1)

# Global faithfulness results table
faithfulness_results = []

print('XAI helper functions defined.')

In [ ]:
# ============================================================
# CELL 7: XAI HELPERS — Faithfulness Metrics (Deletion & Insertion)
# Used by ALL subsequent XAI cells
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch.nn.functional as F

def compute_faithfulness(model, img_tensor, saliency_map, class_idx, device,
                          steps=10, mode='deletion'):
    """
    Compute deletion or insertion faithfulness curve.
    img_tensor: (1, 3, H, W) normalized tensor
    saliency_map: (H, W) numpy array, higher = more important
    Returns: (step_fractions, scores)
    """
    model.eval()
    H, W = img_tensor.shape[-2], img_tensor.shape[-1]
    N = H * W

    # Flatten and sort by importance (descending)
    flat_sal = saliency_map.flatten()
    sorted_idx = np.argsort(flat_sal)[::-1]  # most important first

    # Blurred baseline for insertion
    blurred = F.avg_pool2d(img_tensor, kernel_size=11, stride=1, padding=5)

    fractions = np.linspace(0, 1, steps + 1)
    scores = []

    with torch.no_grad():
        for frac in fractions:
            k = int(frac * N)
            mask = np.zeros(N, dtype=np.float32)
            if k > 0:
                mask[sorted_idx[:k]] = 1.0
            mask_2d = torch.tensor(mask.reshape(1, 1, H, W)).to(device)

            if mode == 'deletion':
                perturbed = img_tensor.to(device) * (1 - mask_2d)
            else:  # insertion
                perturbed = blurred.to(device) * (1 - mask_2d) + img_tensor.to(device) * mask_2d

            out = model(perturbed)
            prob = torch.sigmoid(out[0, class_idx]).item()
            scores.append(prob)

    return fractions.tolist(), scores


def plot_faithfulness(del_fracs, del_scores, ins_fracs, ins_scores,
                       model_name, xai_method, class_name, save_path):
    del_auc = np.trapz(del_scores, del_fracs)
    ins_auc = np.trapz(ins_scores, ins_fracs)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{model_name} | {xai_method} | Class: {class_name}', fontsize=13)

    axes[0].plot(del_fracs, del_scores, 'r-o', markersize=4)
    axes[0].set_title(f'Deletion (AUC={del_auc:.3f})')
    axes[0].set_xlabel('Fraction of pixels removed')
    axes[0].set_ylabel('Probability')

    axes[1].plot(ins_fracs, ins_scores, 'g-o', markersize=4)
    axes[1].set_title(f'Insertion (AUC={ins_auc:.3f})')
    axes[1].set_xlabel('Fraction of pixels revealed')
    axes[1].set_ylabel('Probability')

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved faithfulness: {save_path}')
    return del_auc, ins_auc


def get_sample_image(dataset, idx=0):
    """Return (img_tensor (1,3,H,W), orig_pil, label_vec)"""
    img_tensor, labels = dataset[idx]
    img_tensor = img_tensor.unsqueeze(0)  # (1,3,H,W)
    # Denormalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    orig = img_tensor.squeeze(0) * std + mean
    orig = orig.permute(1, 2, 0).numpy().clip(0, 1)
    return img_tensor, orig, labels


def overlay_heatmap(orig_img, heatmap, alpha=0.5):
    """Overlay a (H,W) heatmap onto orig_img (H,W,3) numpy [0,1]."""
    import matplotlib.cm as cm
    heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heatmap_color = cm.jet(heatmap_norm)[:, :, :3]  # (H,W,3)
    overlay = alpha * heatmap_color + (1 - alpha) * orig_img
    return overlay.clip(0, 1)

# Global faithfulness results table
faithfulness_results = []

print('XAI helper functions defined.')

In [ ]:
# ============================================================
# CELL 8: XAI — GRAD-CAM, GRAD-CAM++, XGRAD-CAM (CNN Models)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import torchvision.models as models
import timm

print(f'Device: {DEVICE}')

# ── Grad-CAM implementation ───────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, img_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(img_tensor)
        # Multi-label: backprop only through target class logit
        logit = output[:, class_idx]
        logit.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # GAP
        cam = (weights * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam


class GradCAMPP:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, img_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(img_tensor)
        logit = output[:, class_idx]
        logit.backward()

        grads = self.gradients  # (1, C, H, W)
        acts  = self.activations  # (1, C, H, W)

        grads_sq = grads ** 2
        grads_cu = grads ** 3
        alpha_num = grads_sq
        alpha_den = 2 * grads_sq + acts * grads_cu.sum(dim=(2, 3), keepdim=True) + 1e-7
        alpha = alpha_num / alpha_den

        weights = (alpha * torch.relu(grads)).sum(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam


class XGradCAM:
    """XGrad-CAM: gradient × activation weighted."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, img_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(img_tensor)
        logit = output[:, class_idx]
        logit.backward()

        acts  = self.activations
        grads = self.gradients
        # XGrad-CAM weights = sum(A_k * grad_k) / (sum(A_k) + eps)
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        weights = (acts * grads).sum(dim=(2, 3), keepdim=True) / (sum_acts + 1e-7)
        cam = (weights * acts).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam


def run_gradcam_xai(model, model_name, target_layer, xai_class, xai_name,
                     dataset, target_class_idx=0, sample_idx=0):
    model.eval()
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    img_tensor = img_tensor.to(DEVICE).requires_grad_(True)

    cam_gen = xai_class(model, target_layer)
    cam = cam_gen.generate(img_tensor, target_class_idx)

    class_name = LABELS[target_class_idx]
    overlay = overlay_heatmap(orig_img, cam)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cam, cmap='jet'); axes[1].set_title(f'{xai_name} CAM'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | {xai_name} | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_{xai_name}_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    # Faithfulness
    img_for_faith = img_tensor.detach()
    d_fracs, d_scores = compute_faithfulness(model, img_for_faith, cam, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_for_faith, cam, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_{xai_name}_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, xai_name, class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': xai_name,
                                  'Deletion AUC': round(del_auc, 4),
                                  'Insertion AUC': round(ins_auc, 4)})
    return cam


# ── Load Models ──────────────────────────────────────────────
# ResNet50
model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()

# DenseNet121
model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()

# EfficientNet-B0
model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()

# Define target layers
rn50_layer   = model_rn50.layer4[-1].conv3
dn121_layer  = model_dn121.features.denseblock4.denselayer16.conv2
efn_layer    = model_efn.conv_head

# ── Run all 3 CAM variants on ResNet50 for each class ────────
TARGET_CLASS = 1  # Diabetes as demo; loop over all classes in production

for xai_cls, xai_nm in [(GradCAM, 'GradCAM'), (GradCAMPP, 'GradCAMPP'), (XGradCAM, 'XGradCAM')]:
    print(f'\n--- {xai_nm} on ResNet50 ---')
    run_gradcam_xai(model_rn50, 'resnet50', rn50_layer, xai_cls, xai_nm,
                    test_dataset, target_class_idx=TARGET_CLASS, sample_idx=0)

    print(f'\n--- {xai_nm} on DenseNet121 ---')
    run_gradcam_xai(model_dn121, 'densenet121', dn121_layer, xai_cls, xai_nm,
                    test_dataset, target_class_idx=TARGET_CLASS, sample_idx=0)

    print(f'\n--- {xai_nm} on EfficientNet-B0 ---')
    run_gradcam_xai(model_efn, 'efficientnet_b0', efn_layer, xai_cls, xai_nm,
                    test_dataset, target_class_idx=TARGET_CLASS, sample_idx=0)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Grad-CAM, Grad-CAM++, XGrad-CAM done.')

In [ ]:
# ============================================================
# CELL 9: XAI — VIT ATTENTION ROLLOUT
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import timm
from pathlib import Path

print(f'Device: {DEVICE}')

# Load ViT model
model_vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
model_vit.load_state_dict(torch.load(OUTPUT_DIR / 'best_vit_base_patch16_224.pth', map_location=DEVICE))
model_vit = model_vit.to(DEVICE).eval()

class AttentionRollout:
    def __init__(self, model, discard_ratio=0.9, head_fusion='mean'):
        self.model = model
        self.discard_ratio = discard_ratio
        self.head_fusion = head_fusion
        self.attention_maps = []
        self._register_hooks()

    def _register_hooks(self):
        for block in self.model.blocks:
            block.attn.register_forward_hook(self._attn_hook)

    def _attn_hook(self, module, input, output):
        # timm ViT attention: output is (B, N, dim)
        # We need to get the attention weights — re-compute
        pass  # handled differently below

    def get_attention_rollout(self, img_tensor):
        self.attention_maps = []

        # Monkey-patch attn forward to capture attention weights
        hooks = []
        def make_hook(attn_maps_list):
            def hook(module, input, output):
                # Recompute attention weights
                B, N, C = input[0].shape
                qkv = module.qkv(input[0]).reshape(B, N, 3, module.num_heads, C // module.num_heads)
                qkv = qkv.permute(2, 0, 3, 1, 4)
                q, k, v = qkv.unbind(0)
                scale = q.shape[-1] ** -0.5
                attn = (q @ k.transpose(-2, -1)) * scale
                attn = attn.softmax(dim=-1)
                attn_maps_list.append(attn.detach().cpu())
            return hook

        for block in self.model.blocks:
            h = block.attn.register_forward_hook(make_hook(self.attention_maps))
            hooks.append(h)

        with torch.no_grad():
            _ = self.model(img_tensor)

        for h in hooks:
            h.remove()

        # Rollout computation
        result = torch.eye(self.attention_maps[0].shape[-1])
        for attn in self.attention_maps:
            if self.head_fusion == 'mean':
                attn_fused = attn.mean(dim=1)  # (B, N, N)
            elif self.head_fusion == 'max':
                attn_fused = attn.max(dim=1)[0]
            else:
                attn_fused = attn.min(dim=1)[0]

            attn_fused = attn_fused[0]  # (N, N)

            # Discard low-attention tokens
            flat = attn_fused.flatten()
            threshold = torch.quantile(flat, self.discard_ratio)
            attn_fused[attn_fused < threshold] = 0

            # Add identity and normalize
            attn_fused = attn_fused + torch.eye(attn_fused.shape[0])
            attn_fused = attn_fused / attn_fused.sum(dim=-1, keepdim=True)

            result = torch.matmul(attn_fused, result)

        # CLS token attention to patches
        mask = result[0, 1:]  # exclude CLS
        num_patches = int(mask.shape[0] ** 0.5)
        mask = mask.reshape(num_patches, num_patches).numpy()
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        import cv2
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        return mask


# Generate attention rollout for each class
rollout = AttentionRollout(model_vit, discard_ratio=0.9)

img_tensor, orig_img, labels = get_sample_image(test_dataset, idx=0)
img_tensor = img_tensor.to(DEVICE)

TARGET_CLASS = 1  # Diabetes
class_name = LABELS[TARGET_CLASS]

attn_map = rollout.get_attention_rollout(img_tensor)
overlay = overlay_heatmap(orig_img, attn_map)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(attn_map, cmap='jet'); axes[1].set_title('Attention Rollout'); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.suptitle(f'ViT | Attention Rollout | Class: {class_name}')
save_img = OUTPUT_DIR / f'vit_base_AttentionRollout_{class_name}.png'
plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
print(f'Saved: {save_img}')

# Faithfulness
d_fracs, d_scores = compute_faithfulness(model_vit, img_tensor.detach(), attn_map, TARGET_CLASS, DEVICE, steps=10, mode='deletion')
i_fracs, i_scores = compute_faithfulness(model_vit, img_tensor.detach(), attn_map, TARGET_CLASS, DEVICE, steps=10, mode='insertion')
faith_path = OUTPUT_DIR / 'vit_base_AttentionRollout_faithfulness.png'
del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                      'vit_base', 'AttentionRollout', class_name, faith_path)
faithfulness_results.append({'Model': 'vit_base', 'XAI': 'AttentionRollout',
                              'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('ViT Attention Rollout done.')

In [ ]:
# ============================================================
# CELL 10: XAI — INTEGRATED GRADIENTS (Captum)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import timm
from captum.attr import IntegratedGradients
from pathlib import Path

print(f'Device: {DEVICE}')

# Load ResNet50
model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()

def run_integrated_gradients(model, model_name, dataset, target_class_idx=1, sample_idx=0, n_steps=50):
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    img_tensor = img_tensor.to(DEVICE)
    baseline = torch.zeros_like(img_tensor).to(DEVICE)  # black baseline

    ig = IntegratedGradients(model)
    attributions = ig.attribute(img_tensor, baseline,
                                 target=target_class_idx,
                                 n_steps=n_steps)
    # Aggregate channels and normalize
    attr = attributions.squeeze().cpu().detach().numpy()  # (3, H, W)
    attr_map = np.abs(attr).sum(axis=0)  # (H, W)
    attr_map = (attr_map - attr_map.min()) / (attr_map.max() - attr_map.min() + 1e-8)

    class_name = LABELS[target_class_idx]
    overlay = overlay_heatmap(orig_img, attr_map)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(attr_map, cmap='hot'); axes[1].set_title('IG Attribution'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | Integrated Gradients | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_IntegratedGradients_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    # Faithfulness
    d_fracs, d_scores = compute_faithfulness(model, img_tensor.detach(), attr_map, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_tensor.detach(), attr_map, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_IntegratedGradients_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, 'IntegratedGradients', class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': 'IntegratedGradients',
                                  'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

# Run on ResNet50, DenseNet121, EfficientNet-B0
TARGET_CLASS = 1
run_integrated_gradients(model_rn50, 'resnet50', test_dataset, target_class_idx=TARGET_CLASS)

model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()
run_integrated_gradients(model_dn121, 'densenet121', test_dataset, target_class_idx=TARGET_CLASS)

model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()
run_integrated_gradients(model_efn, 'efficientnet_b0', test_dataset, target_class_idx=TARGET_CLASS)

model_vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
model_vit.load_state_dict(torch.load(OUTPUT_DIR / 'best_vit_base_patch16_224.pth', map_location=DEVICE))
model_vit = model_vit.to(DEVICE).eval()
run_integrated_gradients(model_vit, 'vit_base', test_dataset, target_class_idx=TARGET_CLASS)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Integrated Gradients done.')

In [ ]:
# ============================================================
# CELL 11: XAI — SMOOTHGRAD
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import timm
from pathlib import Path

print(f'Device: {DEVICE}')

def smoothgrad(model, img_tensor, class_idx, n_samples=50, noise_level=0.15):
    """
    SmoothGrad: average gradients over noisy copies of input.
    img_tensor: (1, 3, H, W) on device
    Returns: (H, W) numpy saliency map
    """
    model.eval()
    std = noise_level * (img_tensor.max() - img_tensor.min()).item()
    smooth_grad = torch.zeros_like(img_tensor)

    for _ in range(n_samples):
        noise = torch.randn_like(img_tensor) * std
        noisy = (img_tensor + noise).detach().requires_grad_(True)
        out = model(noisy)
        logit = out[:, class_idx]
        model.zero_grad()
        logit.backward()
        smooth_grad += noisy.grad.detach()

    smooth_grad = smooth_grad / n_samples
    saliency = smooth_grad.squeeze().abs().cpu().numpy()  # (3, H, W)
    saliency = saliency.sum(axis=0)  # (H, W)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency


def run_smoothgrad(model, model_name, dataset, target_class_idx=1, sample_idx=0):
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    img_tensor = img_tensor.to(DEVICE)
    class_name = LABELS[target_class_idx]

    saliency = smoothgrad(model, img_tensor, target_class_idx)
    overlay = overlay_heatmap(orig_img, saliency)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(saliency, cmap='hot'); axes[1].set_title('SmoothGrad'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | SmoothGrad | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_SmoothGrad_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    d_fracs, d_scores = compute_faithfulness(model, img_tensor.detach(), saliency, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_tensor.detach(), saliency, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_SmoothGrad_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, 'SmoothGrad', class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': 'SmoothGrad',
                                  'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

TARGET_CLASS = 1

model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()
run_smoothgrad(model_rn50, 'resnet50', test_dataset, TARGET_CLASS)

model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()
run_smoothgrad(model_dn121, 'densenet121', test_dataset, TARGET_CLASS)

model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()
run_smoothgrad(model_efn, 'efficientnet_b0', test_dataset, TARGET_CLASS)

model_vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
model_vit.load_state_dict(torch.load(OUTPUT_DIR / 'best_vit_base_patch16_224.pth', map_location=DEVICE))
model_vit = model_vit.to(DEVICE).eval()
run_smoothgrad(model_vit, 'vit_base', test_dataset, TARGET_CLASS)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('SmoothGrad done.')

In [ ]:
# ============================================================
# CELL 12: XAI — DEEP SHAP (Captum DeepLiftShap)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import timm
from captum.attr import DeepLiftShap
from pathlib import Path

print(f'Device: {DEVICE}')

def run_deep_shap(model, model_name, dataset, target_class_idx=1, sample_idx=0, n_background=10):
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    img_tensor = img_tensor.to(DEVICE)
    class_name = LABELS[target_class_idx]

    # Build background from random samples
    bg_tensors = []
    indices = np.random.choice(len(dataset), n_background, replace=False)
    for idx in indices:
        t, _, _ = get_sample_image(dataset, idx)
        bg_tensors.append(t)
    background = torch.cat(bg_tensors, dim=0).to(DEVICE)  # (n_background, 3, H, W)

    dls = DeepLiftShap(model)
    attributions = dls.attribute(img_tensor, background, target=target_class_idx)

    attr = attributions.squeeze().cpu().detach().numpy()  # (3, H, W)
    attr_map = np.abs(attr).sum(axis=0)  # (H, W)
    attr_map = (attr_map - attr_map.min()) / (attr_map.max() - attr_map.min() + 1e-8)

    overlay = overlay_heatmap(orig_img, attr_map)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(attr_map, cmap='RdBu_r'); axes[1].set_title('DeepLIFT SHAP'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | DeepSHAP | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_DeepSHAP_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    d_fracs, d_scores = compute_faithfulness(model, img_tensor.detach(), attr_map, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_tensor.detach(), attr_map, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_DeepSHAP_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, 'DeepSHAP', class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': 'DeepSHAP',
                                  'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

TARGET_CLASS = 1

model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()
run_deep_shap(model_rn50, 'resnet50', test_dataset, TARGET_CLASS)

model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()
run_deep_shap(model_dn121, 'densenet121', test_dataset, TARGET_CLASS)

model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()
run_deep_shap(model_efn, 'efficientnet_b0', test_dataset, TARGET_CLASS)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Deep SHAP done.')

In [ ]:
# ============================================================
# CELL 13: XAI — OCCLUSION SENSITIVITY
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import timm
from pathlib import Path

print(f'Device: {DEVICE}')

def occlusion_sensitivity(model, img_tensor, class_idx, patch_size=32, stride=16):
    """
    Sliding-window occlusion sensitivity.
    Returns (H, W) heatmap showing prediction drop when region is occluded.
    """
    model.eval()
    B, C, H, W = img_tensor.shape

    with torch.no_grad():
        baseline_prob = torch.sigmoid(model(img_tensor.to(DEVICE))[0, class_idx]).item()

    heatmap = np.zeros((H, W), dtype=np.float32)
    count   = np.zeros((H, W), dtype=np.float32)

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            occluded = img_tensor.clone()
            occluded[:, :, y:y+patch_size, x:x+patch_size] = 0.0
            with torch.no_grad():
                prob = torch.sigmoid(model(occluded.to(DEVICE))[0, class_idx]).item()
            drop = baseline_prob - prob  # positive = important region
            heatmap[y:y+patch_size, x:x+patch_size] += drop
            count[y:y+patch_size, x:x+patch_size]   += 1

    heatmap = heatmap / (count + 1e-8)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap


def run_occlusion(model, model_name, dataset, target_class_idx=1, sample_idx=0):
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    img_tensor = img_tensor.to(DEVICE)
    class_name = LABELS[target_class_idx]

    heatmap = occlusion_sensitivity(model, img_tensor, target_class_idx, patch_size=32, stride=16)
    overlay = overlay_heatmap(orig_img, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(heatmap, cmap='hot'); axes[1].set_title('Occlusion Sensitivity'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | Occlusion Sensitivity | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_OcclusionSensitivity_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    d_fracs, d_scores = compute_faithfulness(model, img_tensor.detach(), heatmap, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_tensor.detach(), heatmap, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_OcclusionSensitivity_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, 'OcclusionSensitivity', class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': 'OcclusionSensitivity',
                                  'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

TARGET_CLASS = 1

model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()
run_occlusion(model_rn50, 'resnet50', test_dataset, TARGET_CLASS)

model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()
run_occlusion(model_dn121, 'densenet121', test_dataset, TARGET_CLASS)

model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()
run_occlusion(model_efn, 'efficientnet_b0', test_dataset, TARGET_CLASS)

model_vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
model_vit.load_state_dict(torch.load(OUTPUT_DIR / 'best_vit_base_patch16_224.pth', map_location=DEVICE))
model_vit = model_vit.to(DEVICE).eval()
run_occlusion(model_vit, 'vit_base', test_dataset, TARGET_CLASS)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Occlusion Sensitivity done.')

In [ ]:
# ============================================================
# CELL 14: XAI — RISE (Randomized Input Sampling for Explanation)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as models
import timm
import torch.nn.functional as F
from pathlib import Path

print(f'Device: {DEVICE}')

def generate_rise_masks(n_masks, input_size, cell_size=8, p=0.5):
    """
    Generate random binary masks for RISE.
    Returns: (n_masks, 1, H, W) tensor
    """
    H, W = input_size
    grid_h = H // cell_size + 1
    grid_w = W // cell_size + 1
    masks = []
    for _ in range(n_masks):
        # Random binary grid
        grid = (np.random.rand(grid_h, grid_w) < p).astype(np.float32)
        # Upsample to input size + random shift
        grid_tensor = torch.tensor(grid).unsqueeze(0).unsqueeze(0)  # (1,1,gh,gw)
        upsampled = F.interpolate(grid_tensor, size=(H + cell_size, W + cell_size),
                                  mode='bilinear', align_corners=False)
        # Random crop to (H, W)
        h_off = np.random.randint(0, cell_size + 1)
        w_off = np.random.randint(0, cell_size + 1)
        mask = upsampled[0, 0, h_off:h_off+H, w_off:w_off+W].numpy()
        masks.append(mask)
    return np.stack(masks)[:, np.newaxis, :, :]  # (n_masks, 1, H, W)


def rise(model, img_tensor, class_idx, n_masks=1000, cell_size=8, p=0.5, batch_size=32):
    model.eval()
    _, _, H, W = img_tensor.shape
    masks_np = generate_rise_masks(n_masks, (H, W), cell_size=cell_size, p=p)
    masks_tensor = torch.tensor(masks_np, dtype=torch.float32)  # (n_masks, 1, H, W)

    sal = np.zeros((H, W), dtype=np.float32)
    total_weight = 0.0

    for i in range(0, n_masks, batch_size):
        batch_masks = masks_tensor[i:i+batch_size].to(DEVICE)  # (B, 1, H, W)
        masked_inputs = img_tensor.to(DEVICE) * batch_masks  # (B, 3, H, W)
        with torch.no_grad():
            out = model(masked_inputs)  # (B, num_classes)
            probs = torch.sigmoid(out[:, class_idx]).cpu().numpy()  # (B,)
        for j, w in enumerate(probs):
            sal += w * masks_np[i + j, 0]
            total_weight += w

    sal /= (total_weight + 1e-8)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    return sal


def run_rise(model, model_name, dataset, target_class_idx=1, sample_idx=0):
    img_tensor, orig_img, labels = get_sample_image(dataset, sample_idx)
    class_name = LABELS[target_class_idx]

    sal_map = rise(model, img_tensor, target_class_idx, n_masks=500, cell_size=16, p=0.5)
    overlay = overlay_heatmap(orig_img, sal_map)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(orig_img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(sal_map, cmap='jet'); axes[1].set_title('RISE Saliency'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.suptitle(f'{model_name} | RISE | Class: {class_name}')
    save_img = OUTPUT_DIR / f'{model_name}_RISE_{class_name}.png'
    plt.savefig(save_img, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {save_img}')

    img_tensor_dev = img_tensor.to(DEVICE)
    d_fracs, d_scores = compute_faithfulness(model, img_tensor_dev, sal_map, target_class_idx, DEVICE, steps=10, mode='deletion')
    i_fracs, i_scores = compute_faithfulness(model, img_tensor_dev, sal_map, target_class_idx, DEVICE, steps=10, mode='insertion')
    faith_path = OUTPUT_DIR / f'{model_name}_RISE_faithfulness.png'
    del_auc, ins_auc = plot_faithfulness(d_fracs, d_scores, i_fracs, i_scores,
                                          model_name, 'RISE', class_name, faith_path)
    faithfulness_results.append({'Model': model_name, 'XAI': 'RISE',
                                  'Deletion AUC': round(del_auc, 4), 'Insertion AUC': round(ins_auc, 4)})

TARGET_CLASS = 1

model_rn50 = models.resnet50(weights=None)
model_rn50.fc = nn.Linear(model_rn50.fc.in_features, NUM_CLASSES)
model_rn50.load_state_dict(torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=DEVICE))
model_rn50 = model_rn50.to(DEVICE).eval()
run_rise(model_rn50, 'resnet50', test_dataset, TARGET_CLASS)

model_dn121 = models.densenet121(weights=None)
model_dn121.classifier = nn.Linear(model_dn121.classifier.in_features, NUM_CLASSES)
model_dn121.load_state_dict(torch.load(OUTPUT_DIR / 'best_densenet121.pth', map_location=DEVICE))
model_dn121 = model_dn121.to(DEVICE).eval()
run_rise(model_dn121, 'densenet121', test_dataset, TARGET_CLASS)

model_efn = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
model_efn.load_state_dict(torch.load(OUTPUT_DIR / 'best_efficientnet_b0.pth', map_location=DEVICE))
model_efn = model_efn.to(DEVICE).eval()
run_rise(model_efn, 'efficientnet_b0', test_dataset, TARGET_CLASS)

model_vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
model_vit.load_state_dict(torch.load(OUTPUT_DIR / 'best_vit_base_patch16_224.pth', map_location=DEVICE))
model_vit = model_vit.to(DEVICE).eval()
run_rise(model_vit, 'vit_base', test_dataset, TARGET_CLASS)

if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('RISE done.')

In [ ]:
# ============================================================
# CELL 15: FINAL COMPARISON TABLE
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Model performance summary ────────────────────────────────
model_results = [
    {'Model': 'ResNet50',       'Macro AUC': test_metrics_rn50['macro_auc'],  'Macro mAP': test_metrics_rn50['macro_map'],  'Macro F1': test_metrics_rn50['macro_f1']},
    {'Model': 'DenseNet121',    'Macro AUC': test_metrics_dn121['macro_auc'], 'Macro mAP': test_metrics_dn121['macro_map'], 'Macro F1': test_metrics_dn121['macro_f1']},
    {'Model': 'EfficientNet-B0','Macro AUC': test_metrics_efn['macro_auc'],   'Macro mAP': test_metrics_efn['macro_map'],   'Macro F1': test_metrics_efn['macro_f1']},
    {'Model': 'ViT-Base',       'Macro AUC': test_metrics_vit['macro_auc'],   'Macro mAP': test_metrics_vit['macro_map'],   'Macro F1': test_metrics_vit['macro_f1']},
]
model_df = pd.DataFrame(model_results)
for col in ['Macro AUC', 'Macro mAP', 'Macro F1']:
    model_df[col] = model_df[col].round(4)

print('\n===== MODEL PERFORMANCE COMPARISON =====')
print(model_df.to_string(index=False))

# ── Faithfulness comparison ──────────────────────────────────
faith_df = pd.DataFrame(faithfulness_results)
print('\n===== XAI FAITHFULNESS COMPARISON =====')
print(faith_df.to_string(index=False))

# ── Combined pivot table (for paper) ─────────────────────────
pivot = faith_df.pivot_table(index=['Model', 'XAI'],
                              values=['Deletion AUC', 'Insertion AUC'],
                              aggfunc='mean').round(4)
print('\n===== PIVOT TABLE =====')
print(pivot)

# ── Visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Deletion AUC heatmap
del_pivot = faith_df.pivot(index='XAI', columns='Model', values='Deletion AUC')
sns.heatmap(del_pivot, annot=True, fmt='.3f', cmap='Blues', ax=axes[0])
axes[0].set_title('Deletion AUC (lower = better explanation localization)')

# Insertion AUC heatmap
ins_pivot = faith_df.pivot(index='XAI', columns='Model', values='Insertion AUC')
sns.heatmap(ins_pivot, annot=True, fmt='.3f', cmap='Greens', ax=axes[1])
axes[1].set_title('Insertion AUC (higher = better explanation quality)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'faithfulness_comparison_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Model performance bar chart ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(model_df))
width = 0.25
ax.bar(x - width, model_df['Macro AUC'],  width, label='Macro AUC')
ax.bar(x,         model_df['Macro mAP'],  width, label='Macro mAP')
ax.bar(x + width, model_df['Macro F1'],   width, label='Macro F1')
ax.set_xticks(x); ax.set_xticklabels(model_df['Model'])
ax.set_ylim(0, 1.05); ax.legend(); ax.set_title('Model Comparison on ODIR-5K Test Set')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save tables as CSV
model_df.to_csv(OUTPUT_DIR / 'model_performance.csv', index=False)
faith_df.to_csv(OUTPUT_DIR / 'faithfulness_results.csv', index=False)

print('\nAll results saved. Pipeline complete!')